# 06 — the circuit tier

Circuit resources at full reduced size and for the nested retained instances, the statevector rung on the full lumped operator, and the noise grid: three simulated arms, raw and readout-mitigated per circuit, per instance and recorded step, each read against the pure-noise band of its own instance. `scripts/warm.py --stage 06` simulates one instance per array task into an identity-checked shard; this stage assembles the shards and selects one instance per dataset. Every arm is an Aer simulation; a run on a device, if any, is made by hand from `scripts/export_hardware_instance.py`.

The dataset is the parameters cell: `scripts/run_all.py` sets it for each dataset by replacing that cell in memory and keeps the executed copy as `outputs/executed_notebooks/<stage>_<dataset>.ipynb`; this tracked notebook stays stripped.


In [ ]:
# Parameter cell: scripts/run_all.py replaces it in memory for each dataset.
DATASET = "hmeq"


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path


def repo_root(start: Path | None = None) -> Path:
    """The qmulp checkout this notebook lives in."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "config.py").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("run this notebook from inside the qmulp checkout")


_root = repo_root()
sys.path.insert(0, str(_root / 'src'))
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import polars as pl
from IPython.display import display

from config import DEFAULT_CONFIG
from fingerprint import resolve_config, sha256_path
from paths import ROOT

pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(40)

# The frozen configuration this notebook runs with, echoed so the executed
# notebook records it. No absolute path is printed: artifacts are listed
# relative to the repository root.
config, smoke = resolve_config(DEFAULT_CONFIG)
print("dataset:", DATASET, "| smoke:", smoke)
print(json.dumps(config.as_dict(), indent=1, default=str))


## Run the stage


In [ ]:
from stages import hardware

result = hardware.run(DATASET, config)
display(pl.DataFrame(result["resources"], strict=False).to_pandas())
print(json.dumps(result["statevector"], indent=1, default=str))
print(json.dumps(result["selection"], indent=1, default=str))


## Artifacts


In [ ]:
for path in result["artifacts"]:
    print(f"{sha256_path(path)}  {Path(path).resolve().relative_to(ROOT)}")
